In [1]:
# ============================================================
# EXPERIMENT 5: ZERO-SHOT VS FEW-SHOT PROMPTING
# Task: Customer Review Sentiment Classification
# Groq API + Google Colab
# ============================================================

!pip install -q groq

import os
import re
from getpass import getpass
from groq import Groq

# ------------------------------------------------------------
# STEP 1: Load Groq API Key
# ------------------------------------------------------------

groq_api_key = getpass("Enter your Groq API key: ")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not provided.")

os.environ["GROQ_API_KEY"] = groq_api_key

print("✅ Groq API key loaded successfully.")


# ------------------------------------------------------------
# STEP 2: Create Groq Client
# ------------------------------------------------------------

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ Groq client initialized.")


# ------------------------------------------------------------
# STEP 3: Test Reviews + Ground Truth
# ------------------------------------------------------------

test_reviews = [
    {
        "review": "The battery life is amazing!",
        "actual": "Positive"
    },
    {
        "review": "It works, nothing special.",
        "actual": "Neutral"
    },
    {
        "review": "The camera quality is terrible.",
        "actual": "Negative"
    },
    {
        "review": "I absolutely love this phone!",
        "actual": "Positive"
    },
    {
        "review": "The screen is too small and disappointing.",
        "actual": "Negative"
    }
]


# ------------------------------------------------------------
# STEP 4: Few-Shot Examples
# ------------------------------------------------------------

few_shot_examples = """
Example 1:
Review: "This phone is fantastic and works perfectly."
Sentiment: Positive

Example 2:
Review: "The product is okay, but nothing special."
Sentiment: Neutral

Example 3:
Review: "The product stopped working after one day."
Sentiment: Negative
"""


# ------------------------------------------------------------
# STEP 5: Function to Send Prompt
# ------------------------------------------------------------

def ask_llm(prompt):

    try:

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            max_tokens=20,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:

        return f"ERROR: {e}"


# ------------------------------------------------------------
# STEP 6: Extract Sentiment
# ------------------------------------------------------------

def extract_sentiment(response):

    response_lower = response.lower()

    # Check exact sentiment words
    if "positive" in response_lower:
        return "Positive"

    elif "negative" in response_lower:
        return "Negative"

    elif "neutral" in response_lower:
        return "Neutral"

    else:
        return "Unknown"


# ------------------------------------------------------------
# STEP 7: Store Results
# ------------------------------------------------------------

results = []


# ------------------------------------------------------------
# STEP 8: Run Zero-Shot and Few-Shot Tests
# ------------------------------------------------------------

for i, item in enumerate(test_reviews, start=1):

    review = item["review"]
    actual = item["actual"]

    # --------------------------------------------------------
    # ZERO-SHOT PROMPT
    # --------------------------------------------------------

    zero_shot_prompt = f"""
Classify the following customer review as exactly one of:
Positive, Neutral, or Negative.

Return only the sentiment label.

Review:
"{review}"
"""

    zero_response = ask_llm(zero_shot_prompt)

    zero_label = extract_sentiment(zero_response)


    # --------------------------------------------------------
    # FEW-SHOT PROMPT
    # --------------------------------------------------------

    few_shot_prompt = f"""
Classify the customer review as exactly one of:
Positive, Neutral, or Negative.

Return only the sentiment label.

{few_shot_examples}

Target Review:
"{review}"

Sentiment:
"""

    few_response = ask_llm(few_shot_prompt)

    few_label = extract_sentiment(few_response)


    # --------------------------------------------------------
    # Check Correctness
    # --------------------------------------------------------

    zero_correct = zero_label == actual
    few_correct = few_label == actual

    # Store result
    results.append({
        "review": review,
        "actual": actual,
        "zero_shot": zero_label,
        "few_shot": few_label,
        "zero_correct": zero_correct,
        "few_correct": few_correct
    })


# ============================================================
# STEP 9: Display Individual Results
# ============================================================

print("\n" + "=" * 100)
print("ZERO-SHOT VS FEW-SHOT RESULTS")
print("=" * 100)

for i, result in enumerate(results, start=1):

    print(f"\nReview {i}:")
    print(f"Review       : {result['review']}")
    print(f"Ground Truth : {result['actual']}")
    print(f"Zero-Shot    : {result['zero_shot']}")
    print(f"Few-Shot     : {result['few_shot']}")


# ============================================================
# STEP 10: Calculate Accuracy
# ============================================================

zero_correct_count = sum(
    result["zero_correct"]
    for result in results
)

few_correct_count = sum(
    result["few_correct"]
    for result in results
)

total = len(results)

zero_accuracy = (zero_correct_count / total) * 100
few_accuracy = (few_correct_count / total) * 100


# ============================================================
# STEP 11: Comparison Table
# ============================================================

print("\n" + "=" * 100)
print("COMPARISON TABLE")
print("=" * 100)

print(
    f"{'No.':<5}"
    f"{'Review':<45}"
    f"{'Actual':<12}"
    f"{'Zero-Shot':<12}"
    f"{'Few-Shot':<12}"
)

print("-" * 100)

for i, result in enumerate(results, start=1):

    review_short = result["review"][:42]

    print(
        f"{i:<5}"
        f"{review_short:<45}"
        f"{result['actual']:<12}"
        f"{result['zero_shot']:<12}"
        f"{result['few_shot']:<12}"
    )


# ============================================================
# STEP 12: Accuracy Summary
# ============================================================

print("\n" + "=" * 100)
print("ACCURACY SUMMARY")
print("=" * 100)

print(
    f"Zero-Shot Accuracy : "
    f"{zero_correct_count}/{total} "
    f"({zero_accuracy:.2f}%)"
)

print(
    f"Few-Shot Accuracy  : "
    f"{few_correct_count}/{total} "
    f"({few_accuracy:.2f}%)"
)


# ============================================================
# STEP 13: Improvement
# ============================================================

improvement = few_accuracy - zero_accuracy

print(
    f"\nAccuracy Improvement: "
    f"{improvement:.2f} percentage points"
)


# ============================================================
# STEP 14: Final Analysis
# ============================================================

print("\n" + "=" * 100)
print("FINAL ANALYSIS")
print("=" * 100)

if few_accuracy > zero_accuracy:

    print("""
Few-shot prompting performed better than zero-shot prompting.
Providing examples helped the model understand the expected
classification labels and classification style.
""")

elif few_accuracy == zero_accuracy:

    print("""
Both approaches achieved the same accuracy on this test set.
The few-shot examples may still improve consistency or format
adherence even when accuracy is unchanged.
""")

else:

    print("""
Zero-shot prompting performed better on this particular test set.
Few-shot prompting does not always guarantee higher accuracy.
The quality and relevance of examples are important.
""")

print("✅ Zero-shot vs Few-shot experiment completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00
Enter your Groq API key: ··········
✅ Groq API key loaded successfully.
✅ Groq client initialized.

ZERO-SHOT VS FEW-SHOT RESULTS

Review 1:
Review       : The battery life is amazing!
Ground Truth : Positive
Zero-Shot    : Positive
Few-Shot     : Positive

Review 2:
Review       : It works, nothing special.
Ground Truth : Neutral
Zero-Shot    : Neutral
Few-Shot     : Neutral

Review 3:
Review       : The camera quality is terrible.
Ground Truth : Negative
Zero-Shot    : Negative
Few-Shot     : Negative

Review 4:
Review       : I absolutely love this phone!
Ground Truth : Positive
Zero-Shot    : Positive
Few-Shot     : Positive

Review 5:
Review       : The screen is too small and disappointing.
Ground Truth : Negative
Zero-Shot    : Negative
Few-Shot     : Negative

COMPARISON TABLE
No.  Review                                       Actual      Zero-Shot   Few-Shot    
-------------------------------------